# Cart2Insights — Data Quality Analysis

## Objective

The objective of this notebook is to systematically assess the quality of the
raw e-commerce datasets before performing data cleaning and transformation.

We will investigate:
- Missing values
- Duplicate records
- Primary-key uniqueness
- Foreign-key validity
- Incorrect data types
- Categorical consistency
- Invalid or suspicious values
- Date validity
- Potential outliers


In [15]:
import pandas as pd
import sys
sys.path.insert(0, "../src")
from data_loader import load_datasets

In [16]:
raw = "../data/raw/"
datasets = load_datasets(raw)

In [17]:
customers = datasets["customers"]
orders = datasets["orders"]
order_items = datasets["order_items"]
order_payments = datasets["order_payments"]
order_reviews = datasets["order_reviews"]
products = datasets["products"]
sellers = datasets["sellers"]
geolocation = datasets["geolocation"]
product_category_translation = datasets["product_category_translation"]

### Table : Customers

| Column                     | Meaning                                                      |
| -------------------------- | ------------------------------------------------------------ |
| `customer_id`              | Identifier for a particular customer record                  |
| `customer_unique_id`       | Identifier that represents the actual customer across orders |
| `customer_zip_code_prefix` | Customer's ZIP/postal-code prefix                            |
| `customer_city`            | Customer's city                                              |
| `customer_state`           | Customer's state                                             |


customer_id ≠ customer_unique_id

customer_id identifies the customer record used by the order table, while customer_unique_id lets us recognize the same underlying customer across potentially multiple customer records/orders.

In [18]:
customers["customer_unique_id"].is_unique

False

In [19]:
customers.isna().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

In [20]:
customers.duplicated().sum()

np.int64(0)

In [21]:
customers["customer_id"].nunique()

99441

In [22]:
len(customers)

99441

In [23]:
customers.dtypes

customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

In [24]:
customers["customer_state"].value_counts()

customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
PE     1652
CE     1336
PA      975
MT      907
MA      747
MS      715
PB      536
PI      495
RN      485
AL      413
SE      350
TO      280
RO      253
AM      148
AC       81
AP       68
RR       46
Name: count, dtype: int64

In [25]:
customers["customer_state"].unique()

<ArrowStringArray>
['SP', 'SC', 'MG', 'PR', 'RJ', 'RS', 'PA', 'GO', 'ES', 'BA', 'MA', 'MS', 'CE',
 'DF', 'RN', 'PE', 'MT', 'AM', 'AP', 'AL', 'RO', 'PB', 'TO', 'PI', 'AC', 'SE',
 'RR']
Length: 27, dtype: str

In [26]:
customers["customer_city"].unique()

<ArrowStringArray>
[               'franca', 'sao bernardo do campo',             'sao paulo',
       'mogi das cruzes',              'campinas',        'jaragua do sul',
               'timoteo',              'curitiba',        'belo horizonte',
         'montes claros',
 ...
              'balbinos',          'serra bonita',          'venda branca',
           'sanga puita',               'queiroz',                'siriji',
   'natividade da serra',          'monte bonito',            'sao rafael',
     'eugenio de castro']
Length: 4119, dtype: str

In [27]:
customers["customer_city"].nunique()

4119

In [28]:
customers["customer_city"].value_counts().head(20)

customer_city
sao paulo                15540
rio de janeiro            6882
belo horizonte            2773
brasilia                  2131
curitiba                  1521
campinas                  1444
porto alegre              1379
salvador                  1245
guarulhos                 1189
sao bernardo do campo      938
niteroi                    849
santo andre                797
osasco                     746
santos                     713
goiania                    692
sao jose dos campos        691
fortaleza                  654
sorocaba                   633
recife                     613
florianopolis              570
Name: count, dtype: int64

### Customers Table — Quality Summary

| Check | Finding | Action Required |
|---|---|---|
| Missing values | No missing values | No action |
| Duplicate rows | No exact duplicates | No action |
| Primary key | `customer_id` is unique | No action |
| Data types | No critical issue identified | Review ZIP prefix during cleaning |
| `customer_state` | Values appear consistent | No action |
| `customer_city` | No obvious consistency issue | No action |

### Table: Orders
| Column                          | What it represents                           |
| ------------------------------- | -------------------------------------------- |
| `order_id`                      | Unique identifier for an order               |
| `customer_id`                   | Identifies the customer who placed the order |
| `order_status`                  | Current/status outcome of the order          |
| `order_purchase_timestamp`      | When the order was placed                    |
| `order_approved_at`             | When the order was approved                  |
| `order_delivered_carrier_date`  | When the order was handed to the carrier     |
| `order_delivered_customer_date` | When the order reached the customer          |
| `order_estimated_delivery_date` | Estimated delivery date                      |


In [29]:
orders.isna().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [30]:
orders.isna().mean() * 100

order_id                         0.000000
customer_id                      0.000000
order_status                     0.000000
order_purchase_timestamp         0.000000
order_approved_at                0.160899
order_delivered_carrier_date     1.793023
order_delivered_customer_date    2.981668
order_estimated_delivery_date    0.000000
dtype: float64

| Column                          | Missing | Missing % |
| ------------------------------- | ------: | --------: |
| `order_id`                      |       0 |        0% |
| `customer_id`                   |       0 |        0% |
| `order_status`                  |       0 |        0% |
| `order_purchase_timestamp`      |       0 |        0% |
| `order_approved_at`             |     160 |     0.16% |
| `order_delivered_carrier_date`  |   1,783 |     1.79% |
| `order_delivered_customer_date` |   2,965 |     2.98% |
| `order_estimated_delivery_date` |       0 |        0% |


But maybe no need  to fill or remove them yet.

 order lifecycle date nulls need to be investigated because not every order necessarily reaches every lifecycle stage.

 Purchase -> Approval   ->Carrier   ->Customer delivery

order_status = canceled

order_purchase_timestamp = 2018-01-10

order_approved_at = 2018-01-10

order_delivered_carrier_date = NaN

order_delivered_customer_date = NaN

In [31]:
orders["order_status"].unique()

<ArrowStringArray>
[  'delivered',    'invoiced',     'shipped',  'processing', 'unavailable',
    'canceled',     'created',    'approved']
Length: 8, dtype: str

In [32]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [33]:
orders.groupby("order_status")[
    [
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date"
    ]
].apply(lambda x: x.isna().sum())

,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
order_status,,,
approved,0,2,2
canceled,141,550,619
created,5,5,5
delivered,14,2,8
invoiced,0,314,314
processing,0,301,301
shipped,0,0,1107
unavailable,0,609,609


In [34]:
orders[
    (orders["order_status"] == "delivered") &
    (
        orders["order_approved_at"].isna() |
        orders["order_delivered_carrier_date"].isna() |
        orders["order_delivered_customer_date"].isna()
    )
]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
5323,e04abd8149ef81b95221e88f6ed9ab6a,2127dc6603ac33544953ef05ec155771,delivered,2017-02-18 14:40:00,NaN,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17 00:00:00
16567,8a9adc69528e1001fc68dd0aaebbb54a,4c1ccc74e00993733742a3c786dc3c1f,delivered,2017-02-18 12:45:31,NaN,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21 00:00:00
19031,7013bcfc1c97fe719a7b5e05e61c12db,2941af76d38100e0f8740a374f1a5dc3,delivered,2017-02-18 13:29:47,NaN,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
22663,5cf925b116421afa85ee25e99b4c34fb,29c35fc91fc13fb5073c8f30505d860d,delivered,2017-02-18 16:48:35,NaN,2017-02-22 11:23:10,2017-03-09 07:28:47,2017-03-31 00:00:00
23156,12a95a3c06dbaec84bcfb0e2da5d228a,1e101e0daffaddce8159d25a8e53f2b2,delivered,2017-02-17 13:05:55,NaN,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20 00:00:00
26800,c1d4211b3dae76144deccd6c74144a88,684cb238dc5b5d6366244e0e0776b450,delivered,2017-01-19 12:48:08,NaN,2017-01-25 14:56:50,2017-01-30 18:16:01,2017-03-01 00:00:00
38290,d69e5d356402adc8cf17e08b5033acfb,68d081753ad4fe22fc4d410a9eb1ca01,delivered,2017-02-19 01:28:47,NaN,2017-02-23 03:11:48,2017-03-02 03:41:58,2017-03-27 00:00:00
39334,d77031d6a3c8a52f019764e68f211c69,0bf35cac6cc7327065da879e2d90fae8,delivered,2017-02-18 11:04:19,NaN,2017-02-23 07:23:36,2017-03-02 16:15:23,2017-03-22 00:00:00


In [35]:
delivered_missing_pattern = (
    orders.loc[
        orders["order_status"] == "delivered",
        [
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
        ],
    ]
    .isna()
    .astype(int)
    .value_counts()
)

delivered_missing_pattern

order_approved_at  order_delivered_carrier_date  order_delivered_customer_date
0                  0                             0                                96455
1                  0                             0                                   14
0                  0                             1                                    7
                   1                             0                                    1
                                                 1                                    1
Name: count, dtype: int64

In [36]:
orders.dtypes

order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

In [37]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [38]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

orders[date_columns] = orders[date_columns].apply(pd.to_datetime)

In [39]:
invalid_approval = orders[
    orders["order_approved_at"] < orders["order_purchase_timestamp"]
]

invalid_approval

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date


In [40]:
invalid_approval.shape

(0, 8)

In [41]:
invalid_carrier = orders[
    orders["order_delivered_carrier_date"] < orders["order_purchase_timestamp"]
]

invalid_carrier

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
615,b9afddbdcfadc9a87b41a83271c3e888,85c6af75161b8b2b1af98e82b5a6a5a5,delivered,2018-08-16 13:50:48,2018-08-16 14:05:13,2018-08-16 13:27:00,2018-08-24 14:58:37,2018-09-04
1111,ad133696906f6a78826daa0911b7daec,e6f5b234bb0d847f10eebd70130c5d49,delivered,2018-06-15 15:41:22,2018-06-15 16:19:23,2018-06-15 14:52:00,2018-06-22 18:09:37,2018-07-18
1329,74e033208dc13a7b8127eb8e73d09b76,72fcbb1145f2889eddcba6d6c1d6c090,delivered,2018-05-02 10:48:44,2018-05-02 11:13:45,2018-05-02 09:49:00,2018-05-07 23:06:36,2018-05-29
1372,a6b58794fd2ba533359a76c08df576e3,ccd8e3459ad58ae538c9d2cf35532ba4,delivered,2018-05-14 15:18:23,2018-05-14 15:33:35,2018-05-14 13:46:00,2018-05-19 19:33:32,2018-06-08
1864,5792e0b1c8c8a2bf53af468c9a422c88,5c427f06e7fcf902e309743516e2c580,delivered,2018-07-26 13:25:14,2018-07-26 13:35:14,2018-07-26 12:42:00,2018-07-30 14:45:02,2018-08-09
...,...,...,...,...,...,...,...,...
98172,f7780ea2807db31691e83f0013294035,3088cc4bfafe311f1d84830808c2aa6d,delivered,2018-07-30 15:22:15,2018-07-30 15:35:16,2018-07-30 15:00:00,2018-08-02 18:32:30,2018-08-02
98430,d7646ffe8fdd9e7d9557f9f7cbf04530,f896e70d84f6db9a56b86a47dd8718f0,delivered,2018-05-04 14:50:37,2018-05-04 15:10:22,2018-05-04 14:48:00,2018-05-08 19:06:42,2018-05-16
98672,5ded8a3706eabd813685534724f066de,460316e38d4d241fc79d4f6729f8f7f6,delivered,2018-07-18 08:46:52,2018-07-18 09:01:48,2018-07-18 08:44:00,2018-07-25 13:53:17,2018-08-10
98780,d10046876c7d9f01613da59ffa6cb07f,6559cb5b365f173915f766fefcd86972,delivered,2018-07-18 16:14:16,2018-07-18 16:25:17,2018-07-18 15:34:00,2018-07-23 20:46:44,2018-08-07


In [42]:
invalid_carrier.shape

(166, 8)

In [43]:
invalid_carrier[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date"
    ]
].head(20)

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
615,b9afddbdcfadc9a87b41a83271c3e888,delivered,2018-08-16 13:50:48,2018-08-16 14:05:13,2018-08-16 13:27:00,2018-08-24 14:58:37
1111,ad133696906f6a78826daa0911b7daec,delivered,2018-06-15 15:41:22,2018-06-15 16:19:23,2018-06-15 14:52:00,2018-06-22 18:09:37
1329,74e033208dc13a7b8127eb8e73d09b76,delivered,2018-05-02 10:48:44,2018-05-02 11:13:45,2018-05-02 09:49:00,2018-05-07 23:06:36
1372,a6b58794fd2ba533359a76c08df576e3,delivered,2018-05-14 15:18:23,2018-05-14 15:33:35,2018-05-14 13:46:00,2018-05-19 19:33:32
1864,5792e0b1c8c8a2bf53af468c9a422c88,delivered,2018-07-26 13:25:14,2018-07-26 13:35:14,2018-07-26 12:42:00,2018-07-30 14:45:02
2760,c3eb293fd154223498b6551a728203e8,delivered,2018-07-19 14:06:04,2018-07-19 14:22:51,2018-07-19 13:49:00,2018-07-24 19:35:36
3473,b0c2a7d04b165525254254a728c50a4e,delivered,2018-06-07 13:28:30,2018-06-07 13:57:22,2018-06-07 13:22:00,2018-06-21 17:36:43
3661,2033a4586b5bec3229ebc1675a8ae092,delivered,2018-06-12 10:10:25,2018-06-12 10:39:59,2018-06-12 10:09:00,2018-06-19 14:08:27
4114,08adcddad19d3acf37d1fa01cb9ded1e,delivered,2018-06-27 11:16:44,2018-06-27 11:30:56,2018-06-27 10:57:00,2018-06-29 17:39:53
4159,dee6298ce7d1fb2645141ef9972157aa,shipped,2018-04-30 14:06:12,2018-04-30 14:15:24,2018-04-30 12:59:00,NaT


In [44]:
invalid_carrier["order_status"].value_counts()

order_status
delivered    165
shipped        1
Name: count, dtype: int64

### Chronological Check: Purchase vs Carrier

The carrier delivery timestamp was compared with the order purchase timestamp.

A total of 166 orders were found where the carrier timestamp occurs before
the purchase timestamp.

The issue is highly concentrated among delivered orders:
- 165 delivered orders
- 1 shipped order

These records represent a chronological inconsistency in the supplied
timestamps. They should be investigated and an appropriate handling strategy
should be documented during the cleaning phase.

No records are removed at this stage.

In [45]:
invalid_delivery_sequence = orders[
    orders["order_delivered_customer_date"] <
    orders["order_delivered_carrier_date"]
]

invalid_delivery_sequence.shape

(23, 8)

In [46]:
invalid_delivery_sequence[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
].head(20)

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
6437,a1abeb653a4d4cd1e142ccb8c82cd069,delivered,2017-07-20 11:20:52,2017-07-21 06:43:14,2017-07-28 16:57:58,2017-07-25 19:32:56,2017-08-14
9553,383aa8b2724fe452d9ccd9934a8c628b,delivered,2017-07-02 20:58:43,2017-07-02 21:10:20,2017-07-07 17:22:41,2017-07-06 14:27:51,2017-07-21
13487,cb1134f9010d242e9515ad1c78ec0c39,delivered,2017-07-16 12:35:34,2017-07-18 06:03:50,2017-07-20 19:22:02,2017-07-19 14:13:28,2017-08-08
14474,dceb62e8fa94b46006c9554fed743df0,delivered,2017-07-20 20:58:05,2017-07-22 11:45:11,2017-08-01 18:23:30,2017-07-26 18:09:10,2017-08-11
19268,5f9d46795c3126674e52becb3a1a517f,delivered,2017-07-18 11:48:20,2017-07-18 12:03:29,2017-07-20 23:03:42,2017-07-20 18:52:41,2017-07-31
21338,8c78d01de3a9009e23d6877a7cc9be20,delivered,2016-10-08 15:36:50,2016-10-08 18:13:44,2016-10-26 11:41:53,2016-10-25 17:51:46,2016-11-30
22520,b27af682321527a6349f1761eb3f360c,delivered,2017-06-14 20:17:04,2017-06-14 20:30:08,2017-06-27 14:51:54,2017-06-26 15:45:35,2017-07-14
25393,1cc3ae63caffff2d6c3ee3e78e074acf,delivered,2017-08-07 21:35:22,2017-08-08 21:45:15,2017-08-10 18:28:56,2017-08-10 18:05:38,2017-08-25
25646,e37f11cae9985ca58f0b56f268720537,delivered,2017-07-26 11:46:34,2017-07-27 10:10:16,2017-08-01 18:17:47,2017-07-31 17:49:56,2017-08-24
27470,fa3e37584f4fdb1ded0e0de700dfcb4e,delivered,2017-07-30 19:32:23,2017-07-30 19:45:09,2017-08-09 18:18:43,2017-08-01 21:13:01,2017-08-18


In [47]:
invalid_delivery_sequence["order_status"].value_counts()

order_status
delivered    23
Name: count, dtype: int64

### Chronological Check: Carrier vs Customer Delivery

The customer delivery timestamp was compared with the carrier timestamp.

A total of 23 orders were found where the customer delivery timestamp
occurred before the carrier timestamp.

All 23 records have `order_status = delivered`.

These records represent chronological inconsistencies in the supplied
timestamps and should be considered during the cleaning phase.

No records are removed at this stage.

In [48]:
invalid_customer_delivery = orders[
    orders["order_delivered_customer_date"] <
    orders["order_purchase_timestamp"]
]

invalid_customer_delivery.shape

(0, 8)

### Chronological Check: Purchase vs Customer Delivery

The customer delivery timestamp was compared with the order purchase
timestamp.

No records were found where an order was delivered to the customer before
the order was purchased.

Therefore, no chronological inconsistency was identified between
`order_purchase_timestamp` and `order_delivered_customer_date`.

In [49]:
orders.duplicated().sum()

np.int64(0)

In [50]:
orders["order_id"].nunique()

99441

In [51]:
len(orders)

99441

In [52]:
print(orders["order_id"].nunique()==len(orders))

True


### Primary Key Check

`order_id` is unique across all 99,441 order records.


In [53]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [54]:
orders["order_status"].unique()

<ArrowStringArray>
[  'delivered',    'invoiced',     'shipped',  'processing', 'unavailable',
    'canceled',     'created',    'approved']
Length: 8, dtype: str

### Categorical Consistency Check — `order_status`

The unique values and frequency distribution of `order_status` were inspected.

The column contains eight consistently represented categories:

- delivered
- shipped
- canceled
- unavailable
- invoiced
- processing
- created
- approved

No unexpected or inconsistent category values were identified.

Therefore, no categorical standardization is required for `order_status`
at this stage.

In [55]:
orders[date_columns].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [56]:
orders[date_columns].agg(["min", "max"])

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
min,2016-09-04 21:15:19,2016-09-15 12:16:38,2016-10-08 10:34:01,2016-10-11 13:46:32,2016-09-30
max,2018-10-17 17:30:18,2018-09-03 17:40:06,2018-09-11 19:48:28,2018-10-17 13:22:46,2018-11-12


In [57]:
orders.describe(include="all")

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
count,99441,99441,99441,99441,99281,97658,96476,99441
unique,99441,99441,8,NaN,NaN,NaN,NaN,NaN
top,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,NaN,NaN,NaN,NaN,NaN
freq,1,1,96478,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,NaN,2017-12-31 08:43:12.776581,2017-12-31 18:35:24.098800,2018-01-04 21:49:48.138278,2018-01-14 12:09:19.035542,2018-01-24 03:08:37.730111
min,NaN,NaN,NaN,2016-09-04 21:15:19,2016-09-15 12:16:38,2016-10-08 10:34:01,2016-10-11 13:46:32,2016-09-30 00:00:00
25%,NaN,NaN,NaN,2017-09-12 14:46:19,2017-09-12 23:24:16,2017-09-15 22:28:50.250000,2017-09-25 22:07:22.250000,2017-10-03 00:00:00
50%,NaN,NaN,NaN,2018-01-18 23:04:36,2018-01-19 11:36:13,2018-01-24 16:10:58,2018-02-02 19:28:10.500000,2018-02-15 00:00:00
75%,NaN,NaN,NaN,2018-05-04 15:42:16,2018-05-04 20:35:10,2018-05-08 13:37:45,2018-05-15 22:48:52.250000,2018-05-25 00:00:00
max,NaN,NaN,NaN,2018-10-17 17:30:18,2018-09-03 17:40:06,2018-09-11 19:48:28,2018-10-17 13:22:46,2018-11-12 00:00:00


## Orders Table — Quality Summary

| Check | Finding | Action Required |
|---|---|---|
| Missing values | Missing values in three lifecycle date columns | Investigate and handle during cleaning |
| Exact duplicate rows | None | No action |
| Primary key | `order_id` is unique | No action |
| `customer_id` | No missing values | No action |
| `order_status` | 8 consistent categories | No action |
| Date data types | Lifecycle dates stored as datetime | No action |
| Approval vs purchase | No invalid sequence found | No action |
| Carrier vs purchase | 166 chronological inconsistencies | Investigate during cleaning |
| Customer delivery vs carrier | 23 chronological inconsistencies | Investigate during cleaning |
| Customer delivery vs purchase | No invalid sequence found | No action |
| Date ranges | No obviously unrealistic extreme dates identified | No action |

### Table: Order Items

| Column                | Meaning                                         |
| --------------------- | ----------------------------------------------- |
| `order_id`            | Identifies the order                            |
| `order_item_id`       | Identifies the item number within that order    |
| `product_id`          | Identifies the product purchased                |
| `seller_id`           | Identifies the seller                           |
| `shipping_limit_date` | Shipping-limit timestamp for the item           |
| `price`               | Price of that item                              |
| `freight_value`       | Freight/shipping cost associated with that item |


In [58]:
order_items.isna().sum()

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

In [59]:
order_items.duplicated().sum()

np.int64(0)

In [60]:
order_items["order_item_id"].unique()


array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21])

In [61]:
print(order_items["order_item_id"].nunique() == len(order_items))

False


In [62]:
order_items["order_item_id"].duplicated().sum()

np.int64(112629)

In [63]:
order_items["order_item_id"].value_counts()

order_item_id
1     98666
2      9803
3      2287
4       965
5       460
6       256
7        58
8        36
9        28
10       25
11       17
12       13
13        8
14        7
15        5
16        3
17        3
18        3
19        3
20        3
21        1
Name: count, dtype: int64

In [64]:
order_items[["order_id","order_item_id"]].duplicated().sum()

np.int64(0)

### Composite Key Check

`order_item_id` is not expected to be unique across the entire table because
it identifies the item position within an individual order.

Therefore, the appropriate key is the combination of `order_id` and
`order_item_id`.

The composite key `(order_id, order_item_id)` contains no duplicate
combinations, confirming that each order item can be uniquely identified.

In [65]:
order_items["order_id"].isin(orders["order_id"]).all()

np.True_

### Foreign Key Check — `order_id`

Every `order_id` in the `order_items` table exists in the `orders` table.


In [66]:
order_items["product_id"].isin(products["product_id"]).all()

np.True_

In [67]:
order_items["seller_id"].isin(sellers["seller_id"]).all()

np.True_

In [68]:
order_items.dtypes

order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

| Foreign key                        | Result  |
| ---------------------------------- | ------- |
| `order_id → orders.order_id`       | ✅ Valid |
| `product_id → products.product_id` | ✅ Valid |
| `seller_id → sellers.seller_id`    | ✅ Valid |


In [69]:
order_items[["price", "freight_value"]].describe()

,price,freight_value
count,112650.000000,112650.000000
mean,120.653739,19.990320
std,183.633928,15.806405
min,0.850000,0.000000
25%,39.900000,13.080000
50%,74.990000,16.260000
75%,134.900000,21.150000
max,6735.000000,409.680000


In [70]:
(order_items[["price", "freight_value"]] == 0).sum()

price              0
freight_value    383
dtype: int64

In [71]:
order_items.nlargest(
    10,
    "price"
)[
    [
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "price",
        "freight_value"
    ]
]

,order_id,order_item_id,product_id,seller_id,price,freight_value
3556,0812eb902a67711a1cb742b3cdaa65ae,1,489ae2aa008f021502940f251d4cce7f,e3b4998c7a498169dc7bce44e6bb6277,6735.00,194.31
112233,fefacc66af859508bf1a7934eab1e97f,1,69c590f7ffc7bf8db97190b6cb6ed62e,80ceebb4ee9b31afb6c6a916a574a1e2,6729.00,193.21
107841,f5136e38d1a14a4dbd87dff67da82701,1,1bdf5e6731585cf01aa8169c7028d6ad,ee27a8f15b1dded4d213a468ba4eb391,6499.00,227.66
74336,a96610ab360d42a2e5335a3998b4718a,1,a6492cc69376c469ab6f61d8f44de961,59417c56835dd8e2e72f91f809cd4092,4799.00,151.34
11249,199af31afc78c699f0dbf71fb178d4d4,1,c3ed642d592594bb648ff4a04cee2747,59417c56835dd8e2e72f91f809cd4092,4690.00,74.34
62086,8dbc85d1447242f3b127dda390d56e19,1,259037a6a41845e455183f89c5035f18,c72de06d72748d1a0dfb2125be43ba63,4590.00,91.78
29193,426a9742b533fc6fed17d1fd6d143d7e,1,a1beef8f3992dbd4cd8726796aa69c53,512d298ac2a96d1931b6bd30aa21f61d,4399.87,113.45
45843,68101694e5c5dc7330c91e1bbc36214f,1,6cdf8fc1d741c76586d8b6b15e9eef30,ed4acab38528488b65a9a9c603ff024a,4099.99,75.27
78310,b239ca7cd485940b31882363b52e6674,1,dd113cb02b2af9c8e5787e8f1f0722f6,821fb029fc6e495ca4f08a35d51e53a5,4059.00,104.51
59137,86c4eab1571921a6a6e248ed312f5a5a,1,6902c1962dd19d540807d0ab8fade5c6,fa1c13f2614d7b5c4749cbc52fecda94,3999.90,17.01


In [72]:
order_items["shipping_limit_date"]=order_items["shipping_limit_date"].apply(pd.to_datetime)
#orders[date_columns] = orders[date_columns].apply(pd.to_datetime)

In [73]:
order_items.dtypes

order_id                          str
order_item_id                   int64
product_id                        str
seller_id                         str
shipping_limit_date    datetime64[us]
price                         float64
freight_value                 float64
dtype: object

In [74]:
order_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             112650 non-null  str           
 1   order_item_id        112650 non-null  int64         
 2   product_id           112650 non-null  str           
 3   seller_id            112650 non-null  str           
 4   shipping_limit_date  112650 non-null  datetime64[us]
 5   price                112650 non-null  float64       
 6   freight_value        112650 non-null  float64       
dtypes: datetime64[us](1), float64(2), int64(1), str(3)
memory usage: 16.3 MB


## Order Items Table — Quality Summary

| Check | Finding | Action Required |
|---|---|---|
| Missing values | No missing values | No action |
| Exact duplicate rows | None | No action |
| Composite key | `(order_id, order_item_id)` is unique | No action |
| `order_id` FK | All references valid | No action |
| `product_id` FK | All references valid | No action |
| `seller_id` FK | All references valid | No action |
| Data types | Appropriate after converting `shipping_limit_date` to datetime | No action |
| Negative monetary values | None | No action |
| Zero price | None | No action |
| Zero freight | 383 records | Retain; not confirmed invalid |
| Extreme monetary values | High values present but appear plausible | Retain for analysis |


### Table Order Payments
| Column                 | Meaning                                                 |
| ---------------------- | ------------------------------------------------------- |
| `order_id`             | Identifies which order the payment belongs to           |
| `payment_sequential`   | Sequence number of the payment record within that order |
| `payment_type`         | Method used for payment                                 |
| `payment_installments` | Number of installments used                             |
| `payment_value`        | Value of that payment                                   |


In [75]:
order_payments.info()

<class 'pandas.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  str    
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  str    
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), str(2)
memory usage: 8.1 MB


In [76]:
order_payments.isna().sum()

order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

In [77]:
order_payments.duplicated().sum()

np.int64(0)

In [78]:
order_payments["payment_sequential"].is_unique

False

In [79]:
order_payments["payment_sequential"].nunique()

29

In [80]:
order_payments["payment_sequential"].unique()

array([ 1,  2,  4,  5,  3,  8,  6,  7, 10, 11, 17, 19, 27, 12,  9, 15, 13,
       14, 16, 25, 22, 26, 29, 28, 18, 21, 24, 23, 20])

In [81]:
order_payments["payment_installments"].nunique()

24

In [82]:
order_payments["payment_installments"].unique()

array([ 8,  1,  2,  3,  6,  5,  4, 10,  7, 12,  9, 13, 15, 24, 11, 18, 14,
       20, 21, 17, 22,  0, 16, 23])

In [83]:
order_payments[["order_id", "payment_sequential"]].duplicated().sum()

np.int64(0)

### Composite Key Check

`payment_sequential` is not expected to be unique across the entire table
because the sequence restarts for each order.

The appropriate key is the combination of `order_id` and
`payment_sequential`.

The composite key `(order_id, payment_sequential)` contains no duplicate
combinations, confirming that each payment record can be uniquely identified.

In [84]:
order_payments["order_id"].isin(orders["order_id"]).all()

np.True_

In [85]:
order_payments.dtypes

order_id                    str
payment_sequential        int64
payment_type                str
payment_installments      int64
payment_value           float64
dtype: object

In [86]:
order_payments["payment_type"].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [87]:
order_payments["payment_type"].unique()

<ArrowStringArray>
['credit_card', 'boleto', 'voucher', 'debit_card', 'not_defined']
Length: 5, dtype: str

In [88]:
order_payments["payment_installments"].unique()

array([ 8,  1,  2,  3,  6,  5,  4, 10,  7, 12,  9, 13, 15, 24, 11, 18, 14,
       20, 21, 17, 22,  0, 16, 23])

In [89]:
order_payments["payment_installments"].describe()

count    103886.000000
mean          2.853349
std           2.687051
min           0.000000
25%           1.000000
50%           1.000000
75%           4.000000
max          24.000000
Name: payment_installments, dtype: float64

In [90]:
order_payments["payment_installments"].value_counts().sort_index()

payment_installments
0         2
1     52546
2     12413
3     10461
4      7098
5      5239
6      3920
7      1626
8      4268
9       644
10     5328
11       23
12      133
13       16
14       15
15       74
16        5
17        8
18       27
20       17
21        3
22        1
23        1
24       18
Name: count, dtype: int64

In [91]:
order_payments[
    order_payments["payment_installments"] == 0
]["payment_type"].value_counts()

payment_type
credit_card    2
Name: count, dtype: int64

In [92]:
order_payments[
    order_payments["payment_installments"] == 0
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


In [93]:
order_payments["payment_value"].describe()

count    103886.000000
mean        154.100380
std         217.494064
min           0.000000
25%          56.790000
50%         100.000000
75%         171.837500
max       13664.080000
Name: payment_value, dtype: float64

In [94]:
(order_payments["payment_value"] == 0).sum()

np.int64(9)

In [95]:
order_payments[
    order_payments["payment_value"] == 0
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0


### Investigation of Zero Payment Values

Nine payment records have `payment_value = 0`.

All nine records have one installment. The zero-value records are associated
with the following payment types:

- `voucher`: 6 records
- `not_defined`: 3 records

No zero-value `credit_card` or `debit_card` payment records were identified.


In [96]:
order_payments.nlargest(
    10,
    "payment_value"
)[
    [
        "order_id",
        "payment_sequential",
        "payment_type",
        "payment_installments",
        "payment_value"
    ]
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
52107,03caa2c082116e1d31e67e9ae3700499,1,credit_card,1,13664.08
34370,736e1922ae60d0d6a89247b851902527,1,boleto,1,7274.88
41419,0812eb902a67711a1cb742b3cdaa65ae,1,credit_card,8,6929.31
49581,fefacc66af859508bf1a7934eab1e97f,1,boleto,1,6922.21
85539,f5136e38d1a14a4dbd87dff67da82701,1,boleto,1,6726.66
62409,2cc9089445046817a7539d90805e6e5a,1,boleto,1,6081.54
43232,a96610ab360d42a2e5335a3998b4718a,1,credit_card,10,4950.34
70320,b4c4b76c642808cbe472a32b86cddc95,1,credit_card,5,4809.44
6440,199af31afc78c699f0dbf71fb178d4d4,1,credit_card,8,4764.34
67546,8dbc85d1447242f3b127dda390d56e19,1,credit_card,8,4681.78


In [97]:
order_payments.groupby("order_id").size().describe()

count    99440.000000
mean         1.044710
std          0.381166
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         29.000000
dtype: float64

In [98]:
orders_without_payment = orders[
    ~orders["order_id"].isin(order_payments["order_id"])
]

orders_without_payment

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
30710,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2016-10-04


| Check                                          | Finding                                              | Status                          | Action                               |
| ---------------------------------------------- | ---------------------------------------------------- | ------------------------------- | ------------------------------------ |
| Missing values                                 | No missing values in any column                      | ✅ Clean                         | No action                            |
| Exact duplicate rows                           | 0                                                    | ✅ Clean                         | No action                            |
| `payment_sequential` uniqueness                | Not unique by itself                                 | ✅ Expected                      | No action; sequence resets per order |
| Composite key `(order_id, payment_sequential)` | 0 duplicates                                         | ✅ Clean                         | No action                            |
| `order_id` foreign key                         | All payment `order_id`s exist in `orders`            | ✅ Clean                         | No action                            |
| Data types                                     | Appropriate for all 5 columns                        | ✅ Clean                         | No action                            |
| `payment_type` categories                      | 5 consistent categories                              | ✅ Clean                         | No action                            |
| `payment_installments`                         | 2 records have value `0`, both `credit_card`         | ⚠️ Investigate                  | Flag for cleaning review             |
| Negative `payment_value`                       | 0                                                    | ✅ Clean                         | No action                            |
| Zero `payment_value`                           | 9 records                                            | ⚠️ Investigate                  | Retain for now; don't delete         |
| Extreme `payment_value`                        | Maximum = 13,664.08; inspected and appears plausible | ℹ️ Monitor                      | Retain                               |
| Multiple payments per order                    | Most orders have 1 payment; maximum is 29            | ℹ️ Important structural finding | Account for one-to-many relationship |
| Orders without payment record                  | 1 order has no payment row; order is `delivered`     | ⚠️ Investigate                  | Flag for cleaning review             |


### Table : Order Reviews
| Column                    | Meaning                                       |
| ------------------------- | --------------------------------------------- |
| `review_id`               | Identifier for a review record                |
| `order_id`                | Identifies the order being reviewed           |
| `review_score`            | Customer's review rating                      |
| `review_comment_title`    | Review title                                  |
| `review_comment_message`  | Review text                                   |
| `review_creation_date`    | Date the review was created                   |
| `review_answer_timestamp` | Timestamp associated with the review response |


In [99]:
order_reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   review_id                99224 non-null  str  
 1   order_id                 99224 non-null  str  
 2   review_score             99224 non-null  int64
 3   review_comment_title     11568 non-null  str  
 4   review_comment_message   40977 non-null  str  
 5   review_creation_date     99224 non-null  str  
 6   review_answer_timestamp  99224 non-null  str  
dtypes: int64(1), str(6)
memory usage: 17.8 MB


In [100]:
order_reviews.isna().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [101]:
order_reviews.duplicated().sum()

np.int64(0)

In [102]:
order_reviews["review_id"].is_unique

False

In [103]:
order_reviews["review_id"].duplicated().sum()

np.int64(814)

In [104]:
order_reviews[["review_id","order_id"]].duplicated().sum()

np.int64(0)

In [105]:
order_reviews.dtypes

review_id                    str
order_id                     str
review_score               int64
review_comment_title         str
review_comment_message       str
review_creation_date         str
review_answer_timestamp      str
dtype: object

In [106]:
duplicate_reviews = order_reviews[
    order_reviews["review_id"].duplicated(keep=False)
].sort_values("review_id")

duplicate_reviews.head(20)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44


In [107]:
review_id_order_counts = (
    duplicate_reviews
    .groupby("review_id")["order_id"]
    .nunique()
)

review_id_order_counts.value_counts()

order_id
2    764
3     25
Name: count, dtype: int64

### Investigation of Duplicate `review_id`

`review_id` is not unique in the supplied review data.

A total of 789 review IDs are duplicated:
- 764 duplicated review IDs are associated with 2 different orders.
- 25 duplicated review IDs are associated with 3 different orders.

These duplicated IDs account for 814 additional rows beyond the first
occurrence.

The combination of `review_id` and `order_id` is unique, meaning that there
are no duplicate `(review_id, order_id)` combinations.

Inspection of sample records shows that the same review information can
appear under different orders. The supplied columns do not establish the
reason for this structure.


In [108]:
order_reviews["review_score"].value_counts().sort_index()

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

In [109]:
order_reviews["review_score"].describe()

count    99224.000000
mean         4.086421
std          1.347579
min          1.000000
25%          4.000000
50%          5.000000
75%          5.000000
max          5.000000
Name: review_score, dtype: float64

In [110]:
order_reviews[["review_creation_date","review_answer_timestamp"]] = order_reviews[["review_creation_date","review_answer_timestamp"]].apply(
    pd.to_datetime
)

In [111]:
order_reviews.dtypes


review_id                             str
order_id                              str
review_score                        int64
review_comment_title                  str
review_comment_message                str
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object

In [112]:
invalid_review_dates = order_reviews[
    order_reviews["review_answer_timestamp"] <
    order_reviews["review_creation_date"]
]

invalid_review_dates.shape

(0, 7)

In [113]:
order_reviews["order_id"].isin(orders["order_id"]).all()

np.True_

In [114]:
order_reviews[["review_creation_date", "review_answer_timestamp"]].agg(["min", "max"])

,review_creation_date,review_answer_timestamp
min,2016-10-02,2016-10-07 18:32:28
max,2018-08-31,2018-10-29 12:27:35


In [115]:
purchase_dates = orders.set_index("order_id")["order_purchase_timestamp"]

invalid_review_creation = order_reviews[
    order_reviews["review_creation_date"] <
    order_reviews["order_id"].map(purchase_dates)
]

invalid_review_creation.shape

(74, 7)

In [116]:
invalid_review_creation[
    [
        "order_id",
        "review_score",
        "review_creation_date",
        "review_answer_timestamp"
    ]
].head(20)

,order_id,review_score,review_creation_date,review_answer_timestamp
5293,4d59cc2a2e7bb6c0a851725f5888a9b5,5,2018-07-31,2018-07-31 23:11:57
5503,e5215415bb6f76fe3b7cb68103a0d1c0,1,2016-10-09,2016-10-11 14:31:29
6857,b44b36e6b69be34f58753d3b56d24001,1,2018-08-08,2018-08-08 20:12:05
8404,7813842ae95e8c497fc0233232ae815a,1,2018-08-03,2018-08-06 00:09:52
8486,392ed9afd714e3c74767d0c4d3e3f477,1,2018-08-26,2018-08-28 01:26:38
9020,00b1cb0320190ca0daa2c88b35206009,1,2018-08-28,2018-08-28 18:25:55
10412,0fb9645c128ac84788798f0129496ca7,1,2018-08-04,2018-08-05 17:21:32
10445,2bb02fdf5637a68cd5d13c78c37c3376,2,2018-08-01,2018-08-04 09:51:18
13454,f81e71dd7c044a9ee7c1d6e37d68d64f,1,2018-07-21,2018-07-24 00:09:40
13872,a89abace0dcc01eeb267a9660b5ac126,4,2018-07-31,2018-08-02 15:11:49


In [117]:
(
    order_reviews["review_creation_date"]
    - order_reviews["order_id"].map(purchase_dates)
).loc[invalid_review_creation.index].describe()

count                           74
mean     -19 days +08:07:45.743244
std        21 days 16:57:54.564723
min            -112 days +03:43:58
25%      -27 days +18:31:05.250000
50%             -13 days +00:08:37
75%              -4 days +01:31:42
max              -1 days +15:50:27
dtype: object

In [118]:
invalid_review_creation["review_score"].value_counts().sort_index()


review_score
1    39
2    10
3    10
4     6
5     9
Name: count, dtype: int64

In [119]:
invalid_review_creation["order_id"].map(
    orders.set_index("order_id")["order_status"]
).value_counts()

order_id
canceled     67
delivered     6
shipped       1
Name: count, dtype: int64

In [120]:
order_reviews[
    order_reviews["review_comment_title"].isna() &
    order_reviews["review_comment_message"].isna()
].shape

(56518, 7)

| Check                           | Finding                                  | Status                       |
| ------------------------------- | ---------------------------------------- | ---------------------------- |
| Missing values                  | High missingness in optional text fields | ⚠️ Expected/retain           |
| Exact duplicate rows            | 0                                        | ✅                            |
| `review_id` uniqueness          | 814 duplicate occurrences                | ⚠️ Investigate/document      |
| `(review_id, order_id)`         | 0 duplicates                             | ✅                            |
| `order_id` FK                   | All valid                                | ✅                            |
| `review_score`                  | Values only 1–5                          | ✅                            |
| Review dates                    | Converted to datetime                    | ✅                            |
| Answer before creation          | 0                                        | ✅                            |
| Review creation before purchase | 74                                       | ⚠️ Investigate               |
| Both comment fields missing     | 56,518                                   | ℹ️ Optional text missingness |


## Order Reviews Table — Quality Summary

| Check | Finding | Status | Action |
|---|---|---|---|
| Missing values | `review_comment_title`: 87,656; `review_comment_message`: 58,247 | ⚠️ Investigate | Retain; missing text is not automatically invalid |
| Exact duplicate rows | 0 | ✅ Clean | No action |
| `review_id` uniqueness | 814 duplicate occurrences | ⚠️ Investigate | Retain; do not automatically delete |
| `(review_id, order_id)` uniqueness | 0 duplicates | ✅ Clean | No action |
| `order_id` foreign key | All references valid | ✅ Clean | No action |
| `review_score` range | Values only from 1 to 5 | ✅ Clean | No action |
| Review date types | Date columns converted to datetime | ✅ Clean | No action |
| Answer before creation | 0 records | ✅ Clean | No action |
| Review creation before purchase | 74 records | ⚠️ Investigate | Flag for cleaning review |
| Both review text fields missing | 56,518 records | ℹ️ Expected possibility | Retain; score is still available |
| Review date ranges | No obviously unrealistic extreme dates | ✅ Clean | No action |

### Conclusion

The `order_reviews` table is largely structurally valid, with no missing
review scores, invalid score values, exact duplicate rows, or invalid order
references.

The main data-quality issues are substantial missingness in optional review
text fields, repeated `review_id` values, and 74 records where the recorded
review creation date occurs before the corresponding order purchase timestamp.

The duplicated `review_id` values are associated with different orders, while
`(review_id, order_id)` remains unique. The reason for this structure cannot be
established from the supplied columns.



### Table Products 
| Column                       | Meaning                           |
| ---------------------------- | --------------------------------- |
| `product_id`                 | Unique product identifier         |
| `product_category_name`      | Product category                  |
| `product_name_lenght`        | Length of the product name        |
| `product_description_lenght` | Length of the product description |
| `product_photos_qty`         | Number of product photos          |
| `product_weight_g`           | Product weight in grams           |
| `product_length_cm`          | Product length in centimeters     |
| `product_height_cm`          | Product height in centimeters     |
| `product_width_cm`           | Product width in centimeters      |


In [121]:
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_category_name       32341 non-null  str    
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), str(2)
memory usage: 3.7 MB


In [122]:
products.isna().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [123]:
products["product_id"].is_unique

True

### Primary Key Check

`product_id` is unique across the products table.

Therefore, each product record can be uniquely identified by `product_id`.

In [124]:
products.duplicated().sum()

np.int64(0)

In [125]:
missing_product_info = products[
    products["product_category_name"].isna()
]

missing_product_info

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0
...,...,...,...,...,...,...,...,...,...
32515,b0a0c5dd78e644373b199380612c350a,NaN,NaN,NaN,NaN,1800.0,30.0,20.0,70.0
32589,10dbe0fbaa2c505123c17fdc34a63c56,NaN,NaN,NaN,NaN,800.0,30.0,10.0,23.0
32616,bd2ada37b58ae94cc838b9c0569fecd8,NaN,NaN,NaN,NaN,200.0,21.0,8.0,16.0
32772,fa51e914046aab32764c41356b9d4ea4,NaN,NaN,NaN,NaN,1300.0,45.0,16.0,45.0


In [126]:
missing_product_info[
    [
        "product_category_name",
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty"
    ]
].isna().all(axis=1).value_counts()

True    610
Name: count, dtype: int64

### Missing Value Pattern — Product Information

The 610 records with missing `product_category_name` also have missing
`product_name_lenght`, `product_description_lenght`, and
`product_photos_qty`.

Therefore, these four missing attributes occur together across the same
610 product records.

This indicates a systematic missing-data pattern rather than independent
missing values across the four columns.

In [127]:
missing_product_info[
    [
        "product_id",
        "product_category_name",
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
].head(10)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0
244,e10758160da97891c2fdcbc35f0f031d,NaN,NaN,NaN,NaN,2200.0,16.0,2.0,11.0
294,39e3b9b12cd0bf8ee681bbc1c130feb5,NaN,NaN,NaN,NaN,300.0,16.0,7.0,11.0
299,794de06c32a626a5692ff50e4985d36f,NaN,NaN,NaN,NaN,300.0,18.0,8.0,14.0
347,7af3e2da474486a3519b0cba9dea8ad9,NaN,NaN,NaN,NaN,200.0,22.0,14.0,14.0
428,629beb8e7317703dcc5f35b5463fd20e,NaN,NaN,NaN,NaN,1400.0,25.0,25.0,25.0


### Investigation of the 610 Missing Product Records

The 610 product records with missing category and descriptive attributes still
contain physical product information such as weight and dimensions in the
inspected records.

Therefore, these records are not completely empty product records. The
missingness is concentrated in the category and descriptive fields.

No reason for the missing values can be established from this inspection
alone, so no values are filled or removed during the data-quality analysis.

In [128]:
products[
    products["product_weight_g"].isna()
]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,60.0,865.0,3.0,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Investigation of Missing Physical Attributes

Two products have missing values across all four physical attributes:
`product_weight_g`, `product_length_cm`, `product_height_cm`, and
`product_width_cm`.

The two records show different patterns:

- One product has a valid category, descriptive attributes, and photo count,
  but all physical attributes are missing.
- One product has all non-identifier product attributes missing.

These records are flagged for consideration during the cleaning phase. No
values are filled and no records are removed during data-quality analysis.

In [129]:
products.dtypes

product_id                        str
product_category_name             str
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object

In [130]:
products["product_category_name"].value_counts(dropna=False).head(20)

product_category_name
cama_mesa_banho                      3029
esporte_lazer                        2867
moveis_decoracao                     2657
beleza_saude                         2444
utilidades_domesticas                2335
automotivo                           1900
informatica_acessorios               1639
brinquedos                           1411
relogios_presentes                   1329
telefonia                            1134
bebes                                 919
perfumaria                            868
papelaria                             849
fashion_bolsas_e_acessorios           849
cool_stuff                            789
ferramentas_jardim                    753
pet_shop                              719
NaN                                   610
eletronicos                           517
construcao_ferramentas_construcao     400
Name: count, dtype: int64

In [131]:
numeric_product_columns = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

(products[numeric_product_columns] < 0).sum()

product_name_lenght           0
product_description_lenght    0
product_photos_qty            0
product_weight_g              0
product_length_cm             0
product_height_cm             0
product_width_cm              0
dtype: int64

In [132]:
(products[numeric_product_columns] == 0).sum()


product_name_lenght           0
product_description_lenght    0
product_photos_qty            0
product_weight_g              4
product_length_cm             0
product_height_cm             0
product_width_cm              0
dtype: int64

In [133]:
products[numeric_product_columns].describe()

,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,32341.000000,32341.000000,32341.000000,32949.000000,32949.000000,32949.000000,32949.000000
mean,48.476949,771.495285,2.188986,2276.472488,30.815078,16.937661,23.196728
std,10.245741,635.115225,1.736766,4282.038731,16.914458,13.637554,12.079047
min,5.000000,4.000000,1.000000,0.000000,7.000000,2.000000,6.000000
25%,42.000000,339.000000,1.000000,300.000000,18.000000,8.000000,15.000000
50%,51.000000,595.000000,1.000000,700.000000,25.000000,13.000000,20.000000
75%,57.000000,972.000000,3.000000,1900.000000,38.000000,21.000000,30.000000
max,76.000000,3992.000000,20.000000,40425.000000,105.000000,105.000000,118.000000


In [134]:
products.nlargest(
    10,
    "product_weight_g"
)[
    [
        "product_id",
        "product_category_name",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
]

,product_id,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
25166,26644690fde745fc4654719c3904e1db,cama_mesa_banho,40425.0,13.0,65.0,28.0
344,d0877f0094337c414d23f5a3c7bad20c,moveis_escritorio,30000.0,50.0,50.0,30.0
509,53f92b0474f91fcb5bd188c6a8075c38,utilidades_domesticas,30000.0,76.0,51.0,51.0
955,ceeba7d5636e59173cc5f484e913db3d,NaN,30000.0,65.0,65.0,65.0
1159,f97ad9066c718a6cef93dfcf253d3e0d,moveis_decoracao,30000.0,61.0,70.0,65.0
1469,363a9f5b97bf194da23858be722a7aa5,construcao_ferramentas_construcao,30000.0,100.0,50.0,50.0
1514,dcfeedf441c38e5e7e58ffce194af2bb,beleza_saude,30000.0,67.0,64.0,61.0
1701,1c57458e824ca3d974ec1831a1a55e72,pet_shop,30000.0,79.0,2.0,63.0
1714,c04e948c6900ce99ac47d89b3b6d70cd,cool_stuff,30000.0,70.0,37.0,70.0
1738,0a859d8dc68f6a746b4709217110c439,esporte_lazer,30000.0,90.0,50.0,50.0


In [135]:
for column in ["product_length_cm", "product_height_cm", "product_width_cm"]:
    print(f"\nTop values for {column}")
    print(
        products.nlargest(10, column)[
            [
                "product_id",
                "product_category_name",
                "product_weight_g",
                "product_length_cm",
                "product_height_cm",
                "product_width_cm"
            ]
        ]
    )


Top values for product_length_cm
                            product_id product_category_name  \
88    7f34b85142d1ef9e077a0da9ada27385      moveis_decoracao   
357   f2a1b32f85cad59ff2a8444154ac25f0          climatizacao   
788   11970aff9a8cf29a520127d0d8100901      moveis_decoracao   
1019  d19284bf0893e07e80c26fe4ad33458e      moveis_decoracao   
1151  3c07c4a8d970b0ffad8a97bd6b5e478c      moveis_decoracao   
1403  184e91de5d4a87f023bba45c3e258349      moveis_decoracao   
1614  43506d2b6b5e0535079f88c7dc51c4de    artes_e_artesanato   
2004  34742604f6cd1e891726b849b6890f81         esporte_lazer   
2395  8c34ab7fffe8c063bc8c6da88aa92185         esporte_lazer   
2413  b4b48db403b9792d92330c4a79262763      moveis_decoracao   

      product_weight_g  product_length_cm  product_height_cm  product_width_cm  
88              2300.0              105.0                3.0              70.0  
357             7800.0              105.0               10.0              40.0  
788             25

## Products Table — Quality Summary

| Check | Finding | Status | Action |
|---|---|---|---|
| Missing values | 610 records missing category/descriptive attributes; 2 records missing physical attributes | ⚠️ Investigate | Handle during cleaning |
| Exact duplicate rows | 0 | ✅ Clean | No action |
| Primary key | `product_id` is unique | ✅ Clean | No action |
| Data types | Appropriate for current analysis | ✅ Clean | No action |
| Category consistency | No obvious formatting inconsistencies | ✅ Clean | No action |
| Negative numeric values | None | ✅ Clean | No action |
| Zero numeric values | None | ✅ Clean | No action |
| Extreme weight values | High values observed but no obvious invalid records | ℹ️ Monitor | Retain |
| Extreme dimension values | High values inspected and appear plausible | ℹ️ Monitor | Retain |

### Conclusion

The `products` table is structurally sound, with unique product identifiers,
no exact duplicate rows, and no negative or zero numeric attributes.

The main quality concerns are missing product information in a small number of
records and several statistically extreme but apparently plausible physical
attribute values.



### Table Sellers
| Column                   | Meaning                         |
| ------------------------ | ------------------------------- |
| `seller_id`              | Unique identifier of the seller |
| `seller_zip_code_prefix` | Seller's ZIP/postal-code prefix |
| `seller_city`            | Seller's city                   |
| `seller_state`           | Seller's state                  |


In [136]:
sellers.info()

<class 'pandas.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   seller_id               3095 non-null   str  
 1   seller_zip_code_prefix  3095 non-null   int64
 2   seller_city             3095 non-null   str  
 3   seller_state            3095 non-null   str  
dtypes: int64(1), str(3)
memory usage: 230.3 KB


In [137]:
sellers.isna().sum()

seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

In [138]:
sellers.duplicated().sum()

np.int64(0)

In [139]:
sellers["seller_id"].is_unique

True

In [140]:
sellers.dtypes

seller_id                   str
seller_zip_code_prefix    int64
seller_city                 str
seller_state                str
dtype: object

In [141]:
sellers["seller_state"].value_counts()

seller_state
SP    1849
PR     349
MG     244
SC     190
RJ     171
RS     129
GO      40
DF      30
ES      23
BA      19
CE      13
PE       9
PB       6
RN       5
MS       5
MT       4
RO       2
SE       2
AC       1
PI       1
MA       1
AM       1
PA       1
Name: count, dtype: int64

In [142]:
sellers["seller_state"].unique

<bound method Series.unique of 0       SP
1       SP
2       RJ
3       SP
4       SP
        ..
3090    PR
3091    SC
3092    SP
3093    RS
3094    SP
Name: seller_state, Length: 3095, dtype: str>

In [143]:
sellers["seller_city"].nunique()

611

In [144]:
sellers["seller_city"].value_counts().head(20)

seller_city
sao paulo                694
curitiba                 127
rio de janeiro            96
belo horizonte            68
ribeirao preto            52
guarulhos                 50
ibitinga                  49
santo andre               45
campinas                  41
maringa                   40
sao jose do rio preto     33
sao bernardo do campo     32
osasco                    32
sorocaba                  32
brasilia                  28
porto alegre              28
londrina                  26
goiania                   23
joinville                 22
blumenau                  21
Name: count, dtype: int64

In [145]:
sellers["seller_zip_code_prefix"].nunique()

2246

In [146]:
sellers["seller_zip_code_prefix"].describe()

count     3095.000000
mean     32291.059451
std      32713.453830
min       1001.000000
25%       7093.500000
50%      14940.000000
75%      64552.500000
max      99730.000000
Name: seller_zip_code_prefix, dtype: float64

In [147]:
sellers["seller_zip_code_prefix"].nunique()

2246

In [148]:
seller_zip_state_counts = (
    sellers.groupby("seller_zip_code_prefix")["seller_state"]
    .nunique()
)

seller_zip_state_counts.value_counts().sort_index()

seller_state
1    2229
2      16
3       1
Name: count, dtype: int64

In [149]:
multi_state_zip = seller_zip_state_counts[
    seller_zip_state_counts > 1
]

multi_state_zip

seller_zip_code_prefix
21210    2
22783    2
31160    2
36010    2
37540    2
37795    2
44600    2
80240    2
81020    2
81560    2
83020    2
83321    2
85960    3
88075    2
88301    2
89052    2
95076    2
Name: seller_state, dtype: int64

In [150]:
sellers[
    sellers["seller_zip_code_prefix"].isin(multi_state_zip.index)
].sort_values("seller_zip_code_prefix")

,seller_id,seller_zip_code_prefix,seller_city,seller_state
114,392f7f2c797e4dc077e4311bde2ab8ce,21210,rio de janeiro,RN
1591,95e03ca3d4146e4011985981aeb959b9,21210,rio de janeiro,RJ
364,c716e0b86ed568878475b60fbb6323ad,22783,rio de janeiro,SP
299,b026643914f651e8371425e5b8a1e780,22783,rio de janeiro,RJ
1693,2c00c85d30361cd2ced2969cffbbffa3,31160,belo horizonte,MG
3036,6b9b80d53ba3676eafe60268a810b5a1,31160,belo horizonte,SP
2509,29e6e05b65f93bc224c644319dc52ecf,36010,juiz de fora,MG
680,d9d43faf741ceaa8da52fdbb81d9e0e3,36010,juiz de fora,SP
3012,f789d2c4f2c2eb38fc4373e7a4b35264,36010,juiz de fora,MG
1299,57df9869a600bd6b7c405f2a862eccfb,37540,santa rita do sapucai,MG


## Sellers Table — Quality Summary

| Check | Finding | Status | Action |
|---|---|---|---|
| Missing values | No missing values in any column | ✅ Clean | No action |
| Exact duplicate rows | 0 | ✅ Clean | No action |
| Primary key | `seller_id` is unique | ✅ Clean | No action |
| Data types | Appropriate for the columns | ✅ Clean | No action |
| `seller_state` consistency | State codes consistently formatted | ✅ Clean | No action |
| `seller_city` consistency | No obvious inconsistencies in inspected values | ✅ Clean | No action |
| ZIP-code range | Values from 1,001 to 99,730 | ✅ Plausible | No action |
| ZIP-code uniqueness | 2,246 unique prefixes across 3,095 sellers | ✅ Expected | No action |
| ZIP prefix vs state | 17 ZIP prefixes associated with multiple states | ⚠️ Investigate | Flag for further geographic validation |
| Seller ZIP prefix type | Stored as `int64` | ℹ️ Review | Consider as geographic code during cleaning |

### Conclusion

The `sellers` table is largely clean and structurally sound. It contains no
missing values or exact duplicate rows, and `seller_id` uniquely identifies
each seller.

The main issue identified is that 17 ZIP-code prefixes are associated with
multiple states. These records were inspected and the available seller data
does not provide enough evidence to determine the correct state. They will
therefore be investigated further using geographic information rather than
being modified automatically.

No records are removed or modified during the data-quality analysis.

### Table Geolocation
| Column                        | Meaning                |
| ----------------------------- | ---------------------- |
| `geolocation_zip_code_prefix` | ZIP/postal-code prefix |
| `geolocation_lat`             | Latitude               |
| `geolocation_lng`             | Longitude              |
| `geolocation_city`            | City                   |
| `geolocation_state`           | State                  |


In [151]:
geolocation.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  str    
 4   geolocation_state            1000163 non-null  str    
dtypes: float64(2), int64(1), str(2)
memory usage: 50.1 MB


In [152]:
geolocation.isna().sum()

geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

In [153]:
geolocation.duplicated().sum()

np.int64(261831)

In [154]:
geolocation["geolocation_zip_code_prefix"].is_unique

False

In [155]:
geolocation["geolocation_zip_code_prefix"].duplicated().sum()

np.int64(981148)

In [156]:
261831 / len(geolocation) * 100

26.17883285024541

In [157]:
geolocation[geolocation.duplicated(keep=False)].head(10)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
6,1047,-23.546273,-46.641225,sao paulo,SP
7,1013,-23.546923,-46.634264,sao paulo,SP
8,1029,-23.543769,-46.634278,sao paulo,SP
9,1011,-23.547640,-46.636032,sao paulo,SP
10,1013,-23.547325,-46.634184,sao paulo,SP
13,1012,-23.548946,-46.634671,sao paulo,SP
15,1046,-23.546081,-46.644820,sao paulo,SP


### Exact Duplicate Investigation

The `geolocation` table contains 261,831 exact duplicate rows,
representing approximately 26.18% of the full dataset.

Inspection confirms that some ZIP prefixes legitimately have multiple
geolocation records with different coordinates. These repeated ZIP prefixes
are part of the table's structure and are different from exact duplicate
records.

The exact duplicate rows are therefore flagged as a data-quality issue,
while repeated ZIP prefixes are retained as part of the table's observed
grain.

No records are removed during the data-quality analysis stage.

In [158]:
geolocation.dtypes

geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str
dtype: object

In [159]:
(geolocation["geolocation_lat"].abs() > 90).sum()

np.int64(0)

In [160]:
(geolocation["geolocation_lng"].abs() > 180).sum()

np.int64(0)

In [161]:
geolocation["geolocation_state"].value_counts()


geolocation_state
SP    404268
MG    126336
RJ    121169
RS     61851
PR     57859
SC     38328
BA     36045
GO     20139
ES     16748
PE     16432
DF     12986
MT     12031
CE     11674
PA     10853
MS     10431
MA      7853
PB      5538
RN      5041
PI      4549
AL      4183
TO      3576
SE      3563
RO      3478
AM      2432
AC      1301
AP       853
RR       646
Name: count, dtype: int64

In [162]:
geolocation["geolocation_city"].nunique()

8011

In [163]:
geolocation[["geolocation_zip_code_prefix", "geolocation_city", "geolocation_state"]].drop_duplicates().head(20)

,geolocation_zip_code_prefix,geolocation_city,geolocation_state
0,1037,sao paulo,SP
1,1046,sao paulo,SP
3,1041,sao paulo,SP
4,1035,sao paulo,SP
5,1012,são paulo,SP
6,1047,sao paulo,SP
7,1013,sao paulo,SP
8,1029,sao paulo,SP
9,1011,sao paulo,SP
11,1032,sao paulo,SP


In [164]:
geolocation.groupby(
    ["geolocation_zip_code_prefix", "geolocation_state"]
)["geolocation_city"].nunique().sort_values(ascending=False).head(20)

geolocation_zip_code_prefix  geolocation_state
78290                        MT                   5
13318                        SP                   5
13457                        SP                   5
13454                        SP                   5
6900                         SP                   5
17970                        SP                   5
13455                        SP                   5
28950                        RJ                   5
42850                        BA                   5
87365                        PR                   4
13456                        SP                   4
72270                        DF                   4
76958                        RO                   4
72231                        DF                   4
72220                        DF                   4
89610                        SC                   4
85575                        PR                   4
13855                        SP                   4
9780             

In [165]:
geolocation[
    geolocation["geolocation_city"].str.normalize("NFKD")
    .str.encode("ascii", errors="ignore")
    .str.decode("utf-8")
    .ne(geolocation["geolocation_city"])
]["geolocation_city"].nunique()

2085

In [166]:
zip_city_variants = (
    geolocation.groupby(
        ["geolocation_zip_code_prefix", "geolocation_state"]
    )["geolocation_city"]
    .unique()
)

zip_city_variants.head(20)

geolocation_zip_code_prefix  geolocation_state
1001                         SP                   [sao paulo, são paulo]
1002                         SP                   [sao paulo, são paulo]
1003                         SP                   [sao paulo, são paulo]
1004                         SP                   [sao paulo, são paulo]
1005                         SP                   [sao paulo, são paulo]
1006                         SP                   [sao paulo, são paulo]
1007                         SP                   [sao paulo, são paulo]
1008                         SP                   [sao paulo, são paulo]
1009                         SP                   [sao paulo, são paulo]
1010                         SP                   [sao paulo, são paulo]
1011                         SP                   [sao paulo, são paulo]
1012                         SP                   [são paulo, sao paulo]
1013                         SP                   [sao paulo, são paulo]
1014

geolocation_city contains naming variations such as sao paulo and são paulo, causing the same geographic location to appear as multiple categorical values. This should be considered during categorical standardization in the cleaning phase.

| Check                 | Finding                                                                                                                                        | Status                   | Action                 |
| --------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------- | ------------------------ | ---------------------- |
| Missing values        | No missing values in any column                                                                                                                | ✅ Good                   | No action              |
| Exact duplicates      | 261,831 exact duplicate rows (~26.18%)                                                                                                         | ⚠️ Present               | Review during cleaning |
| ZIP prefix uniqueness | ZIP prefix is not unique                                                                                                                       | ✅ Expected               | No action              |
| Latitude validity     | 0 values outside `[-90, 90]`                                                                                                                   | ✅ Valid                  | No action              |
| Longitude validity    | 0 values outside `[-180, 180]`                                                                                                                 | ✅ Valid                  | No action              |
| State consistency     | 27 two-letter state codes; no obvious formatting issue                                                                                         | ✅ Consistent             | No action              |
| City cardinality      | 8,011 distinct city values                                                                                                                     | ℹ️ Observed              | No action by itself    |
| City naming variation | 2,085 distinct city values contain accented characters                                                                                         | ⚠️ Standardization issue | Review during cleaning |
| ZIP + state → city    | Multiple city labels observed for some ZIP/state combinations; examples are mainly accented/unaccented forms such as `sao paulo` / `são paulo` | ⚠️ Needs standardization | Review during cleaning |


### Table Translation

In [167]:
product_category_translation.info()

<class 'pandas.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   product_category_name          71 non-null     str  
 1   product_category_name_english  71 non-null     str  
dtypes: str(2)
memory usage: 3.5 KB


In [168]:
product_category_translation.isna().sum()

product_category_name            0
product_category_name_english    0
dtype: int64

In [169]:
product_category_translation.duplicated().sum()

np.int64(0)

In [170]:
product_category_translation["product_category_name"].is_unique

True

In [171]:
product_category_translation["product_category_name_english"].is_unique

True

In [172]:
product_category_translation["product_category_name"].nunique

<bound method IndexOpsMixin.nunique of 0                      beleza_saude
1            informatica_acessorios
2                        automotivo
3                   cama_mesa_banho
4                  moveis_decoracao
                  ...              
66                           flores
67               artes_e_artesanato
68                  fraldas_higiene
69    fashion_roupa_infanto_juvenil
70               seguros_e_servicos
Name: product_category_name, Length: 71, dtype: str>

In [173]:
product_category_translation["product_category_name_english"].nunique

<bound method IndexOpsMixin.nunique of 0                 health_beauty
1         computers_accessories
2                          auto
3                bed_bath_table
4               furniture_decor
                ...            
66                      flowers
67        arts_and_craftmanship
68          diapers_and_hygiene
69    fashion_childrens_clothes
70        security_and_services
Name: product_category_name_english, Length: 71, dtype: str>

| Area                    | Key Finding                                                                                              | Status                        | Cleaning Implication                                                  |
| ----------------------- | -------------------------------------------------------------------------------------------------------- | ----------------------------- | --------------------------------------------------------------------- |
| Missing values          | Missingness exists mainly in order lifecycle dates, review text fields, and selected product attributes  | ⚠️ Needs context              | Treat based on business meaning; do not blindly remove rows           |
| Exact duplicates        | Exact duplicate rows were found in `geolocation` (261,831 rows)                                          | ⚠️ Needs review               | Investigate/deduplicate appropriately during cleaning                 |
| Primary/composite keys  | Required keys are unique except `review_id`, which has duplicate values in the supplied review data      | ⚠️ Needs handling             | Use the appropriate grain/key when working with reviews               |
| Foreign keys            | Checked FK relationships have 100% matching parent keys                                                  | ✅ Valid                       | No FK correction required                                             |
| Data types              | Date/time columns identified and converted to datetime for analysis; other types are broadly appropriate | ✅ Mostly valid                | Final type standardization during cleaning                            |
| Invalid numeric values  | No negative monetary values or negative product measurements identified                                  | ✅ Good                        | No removal required based on these checks                             |
| Zero values             | Some zero payment/freight values exist                                                                   | ⚠️ Needs context              | Retain unless business rules establish them as invalid                |
| Date consistency        | Some lifecycle dates occur in unexpected chronological order                                             | ⚠️ Needs investigation        | Handle/document during cleaning rather than blindly deleting          |
| Categorical consistency | Most categorical values are consistent; city-name variations exist in geolocation                        | ⚠️ Standardization needed     | Standardize text values during cleaning                               |
| Geolocation structure   | ZIP prefixes repeat across many rows and exact duplicates are present                                    | ℹ️ Structural + quality issue | Do not treat geolocation as one-row-per-ZIP without aggregation logic |
| Extreme values          | High values in price, payment, weight, etc. were inspected and appear plausible                          | ✅ Retained                    | Do not remove merely because they are extreme                         |
